In [1]:
# put at top
import tempfile, shutil, zipfile, time, os

def atomic_save(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=os.path.dirname(path))
    os.close(fd)
    torch.save(state, tmp)
    shutil.move(tmp, path)  # atomic on same filesystem

def add_to_zip(zip_path, file_path):
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)
    with zipfile.ZipFile(zip_path, mode='a', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(file_path, arcname=os.path.basename(file_path))


In [2]:
# Kaggle-friendly stack
!pip -q install "numpy==1.26.4" "scipy==1.11.4" "scikit-learn==1.3.2" "protobuf>=4.25.1" --upgrade

# install our deps WITHOUT dragging more stuff (prevents re-upgrading numpy/scipy)
!pip -q install --no-deps opencv-python-headless scikit-image timm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 47.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.0/322.0 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<4.0.0dev,>=3.19.5, but you have protobuf 6.32.1 which is incompatible.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.32.1 which is incompatible.
googl

In [3]:
%%writefile p3_stage2b_cmf.py
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet50

# --- helpers ---
class GlobalAvgPool(nn.Module):
    def forward(self, x): return x.mean(dim=(2,3))

# --- FFT adapter (magnitude only) ---
class FFTMagAdapter(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.reduce = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        X = torch.fft.fft2(x, dim=(-2,-1))
        mag = torch.abs(X)
        mag = torch.log1p(mag)
        mean = mag.mean(dim=(2,3), keepdim=True)
        std  = mag.std(dim=(2,3), keepdim=True) + 1e-6
        mag = (mag - mean) / std
        f = self.reduce(mag); f = self.bn(f); f = self.act(f)
        return f  # (B,out_ch,H,W)

# --- Cross-Modal Fusion: Q from RGB, K/V from Freq ---
class CMF(nn.Module):
    def __init__(self, in_ch, d_model=128, heads=2, dropout=0.0):
        super().__init__()
        self.rgb_proj  = nn.Conv2d(in_ch, d_model, 1, bias=False)
        self.freq_proj = nn.Conv2d(in_ch, d_model, 1, bias=False)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True, dropout=dropout)
        self.out = nn.Conv2d(d_model, in_ch, 1, bias=False)
        self.out_bn = nn.BatchNorm2d(in_ch)
    def forward(self, rgb_map, freq_map):
        B,C,H,W = rgb_map.shape
        q = self.rgb_proj(rgb_map).flatten(2).transpose(1,2)   # (B,HW,d)
        k = self.freq_proj(freq_map).flatten(2).transpose(1,2) # (B,HW,d)
        v = k
        fused, _ = self.attn(q, k, v)                          # (B,HW,d)
        fused = fused.transpose(1,2).reshape(B, -1, H, W)
        fused = self.out_bn(self.out(fused))
        return fused  # (B,C,H,W)

# --- tiny encoder (kept frozen initially) ---
class TinyTransformerEncoder(nn.Module):
    def __init__(self, dim=1024, depth=2, heads=8, mlp_ratio=2.0, dropout=0.0):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=int(dim*mlp_ratio),
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=depth)
    def forward(self, x, src_key_padding_mask=None):
        return self.enc(x, src_key_padding_mask=src_key_padding_mask)

# --- SLIC token pooling ---
class SPTokenPool(nn.Module):
    def forward(self, feat_map, seg):  # feat_map: (B,C,H,W), seg: (B,H0,W0) ints
        B,C,H,W = feat_map.shape
        seg = F.interpolate(seg.unsqueeze(1).float(), size=(H,W), mode='nearest').squeeze(1).long()
        toks_list, counts = [], []
        for b in range(B):
            ids = torch.unique(seg[b])
            feats = feat_map[b]
            toks = []
            for rid in ids:
                m = (seg[b]==rid).float()
                w = m / (m.sum()+1e-6)
                toks.append((feats * w).sum(dim=(1,2)))
            toks = torch.stack(toks, 0)           # (R,C)
            toks_list.append(toks); counts.append(toks.size(0))
        maxR = max(counts)
        padded = []
        for t in toks_list:
            if t.size(0)<maxR:
                pad = torch.zeros(maxR - t.size(0), t.size(1), device=t.device, dtype=t.dtype)
                t = torch.cat([t, pad], 0)
            padded.append(t)
        toks = torch.stack(padded, 0)             # (B,R,C)
        mask = torch.arange(maxR, device=toks.device)[None,:] >= torch.tensor(counts, device=toks.device)[:,None]
        return toks, mask

# --- full Stage-2B model ---
class P3Stage2B(nn.Module):
    def __init__(self, n_segments=50, freeze_backbone=True, freeze_encoder=True, d_model=128, heads=2):
        super().__init__()
        self.backbone = resnet50(weights=None)
        self.backbone.fc = nn.Identity()
        self.neck = nn.Conv2d(2048, 1024, 1, bias=False)
        self.neck_bn = nn.BatchNorm2d(1024); self.neck_act = nn.ReLU(inplace=True)
        self.fft = FFTMagAdapter(1024, 1024)
        self.cmf = CMF(1024, d_model=d_model, heads=heads)
        self.pool_sp = SPTokenPool()
        self.encoder = TinyTransformerEncoder(dim=1024, depth=2, heads=8, mlp_ratio=2.0)
        self.head = nn.Sequential(nn.Linear(1024, 512), nn.ReLU(inplace=True), nn.Linear(512, 2))

        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad = False
            for p in self.neck.parameters(): p.requires_grad = False
            for p in self.neck_bn.parameters(): p.requires_grad = False
        if freeze_encoder:
            for p in self.encoder.parameters(): p.requires_grad = False

    def forward(self, x, seg):
        # backbone
        x = self.backbone.conv1(x); x = self.backbone.bn1(x); x = self.backbone.relu(x); x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x); x = self.backbone.layer2(x); x = self.backbone.layer3(x); x = self.backbone.layer4(x)
        h = self.neck_act(self.neck_bn(self.neck(x)))           # (B,1024,h,w)

        # frequency + CMF (residual)
        f = self.fft(h.detach())                                # keep backbone frozen signal
        fused = h + self.cmf(h, f)                              # (B,1024,h,w)

        # superpixel tokens -> tiny encoder -> head
        toks, pad_mask = self.pool_sp(fused, seg)               # (B,R,1024), (B,R)
        enc = self.encoder(toks, src_key_padding_mask=pad_mask) # (B,R,1024)
        enc = enc.mean(dim=1)
        logits = self.head(enc)                                 # (B,2)
        return logits

# --- light-unfreeze helpers for Stage-2B ---
def unfreeze_layer4(model: P3Stage2B):
    for p in model.backbone.layer4.parameters():
        p.requires_grad = True

def unfreeze_last_encoder_block(model: P3Stage2B):
    # unfreeze the last of the tiny encoder layers
    for p in model.encoder.enc.layers[-1].parameters():
        p.requires_grad = True


Writing p3_stage2b_cmf.py


In [4]:
%%writefile run_stage2b.py
import argparse, os, shutil, tempfile, traceback
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
import numpy as np
from skimage.segmentation import slic

from p3_stage2b_cmf import P3Stage2B, unfreeze_layer4

# ---------- utils ----------
def atomic_save(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=os.path.dirname(path))
    os.close(fd)
    torch.save(state, tmp_path)
    shutil.move(tmp_path, path)

def zip_ckpts():
    # create/refresh a zip with all stage2b ckpts
    os.system("zip -j -q /kaggle/working/stage2b_ckpts.zip /kaggle/working/p3_stage2b_*.pth 2>/dev/null || true")

@torch.no_grad()
def accuracy(logits, y):
    return (logits.argmax(1)==y).float().mean().item()

def log_line(s):
    print(s, flush=True)
    with open("/kaggle/working/train_log.txt","a") as f:
        f.write(s+"\n")

# ---------- dataset with SLIC ----------
class SuperpixelImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None, n_segments=50, compactness=10.0, size=320):
        super().__init__(root, transform=transform)
        self.n_segments = n_segments
        self.compactness = compactness
        self.denorm_mean = torch.tensor([0.485,0.456,0.406])[:,None,None]
        self.denorm_std  = torch.tensor([0.229,0.224,0.225])[:,None,None]
    def __getitem__(self, i):
        x,y = super().__getitem__(i)
        x_vis = (x*self.denorm_std + self.denorm_mean).clamp(0,1).permute(1,2,0).cpu().numpy()
        seg = slic(x_vis, n_segments=self.n_segments, compactness=self.compactness, start_label=0).astype(np.int32)
        return x, torch.from_numpy(seg), y

def sp_collate(batch):
    xs, segs, ys = zip(*batch)
    return torch.stack(xs,0), torch.stack(segs,0), torch.tensor(ys, dtype=torch.long)

def make_loaders(root, batch_size=16, size=320, n_segments=50):
    tf_train = transforms.Compose([
        transforms.Resize((size,size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    tf_eval = transforms.Compose([
        transforms.Resize((size,size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    tr = SuperpixelImageFolder(os.path.join(root,"train"), transform=tf_train, n_segments=n_segments)
    va = SuperpixelImageFolder(os.path.join(root,"valid"), transform=tf_eval,  n_segments=n_segments)
    te = SuperpixelImageFolder(os.path.join(root,"test"),  transform=tf_eval,  n_segments=n_segments)
    dl_tr = DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=sp_collate)
    dl_va = DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=sp_collate)
    dl_te = DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=sp_collate)
    return dl_tr, dl_va, dl_te

def train_one_epoch(model, dl, opt, device):
    model.train()
    ce = nn.CrossEntropyLoss()
    totL=totA=n=0
    for x,seg,y in dl:
        x,seg,y = x.to(device), seg.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(x, seg)
        loss = ce(logits, y)
        loss.backward(); opt.step()
        b = x.size(0); totL += loss.item()*b; totA += accuracy(logits.detach(), y)*b; n += b
    return totL/n, totA/n

@torch.no_grad()
def evaluate(model, dl, device):
    model.eval()
    ce = nn.CrossEntropyLoss()
    totL=totA=n=0
    for x,seg,y in dl:
        x,seg,y = x.to(device), seg.to(device), y.to(device)
        logits = model(x, seg)
        loss = ce(logits, y)
        b = x.size(0); totL += loss.item()*b; totA += accuracy(logits, y)*b; n += b
    return totL/n, totA/n

def load_shape_compatible(m, path):
    ckpt = torch.load(path, map_location='cpu')
    if isinstance(ckpt, dict) and "model" in ckpt: ckpt = ckpt["model"]
    msd = m.state_dict()
    keep = {k:v for k,v in ckpt.items() if k in msd and v.shape == msd[k].shape}
    m.load_state_dict(keep, strict=False)
    log_line(f"Loaded {len(keep)} tensors; skipped {len(list(set(ckpt.keys())-set(keep.keys())))} (shape mismatch).")

def main():
    p = argparse.ArgumentParser("Stage-2B safe runner (CMF+SLIC) with atomic saves + autosave zip + auto-extend")
    p.add_argument('--data_root', type=str, required=True)
    p.add_argument('--batch_size', type=int, default=16)
    p.add_argument('--epochs', type=int, default=2)   # base epochs; may auto-extend by +1
    p.add_argument('--lr', type=float, default=5e-4)
    p.add_argument('--n_segments', type=int, default=50)
    p.add_argument('--freeze_backbone', action='store_true')
    p.add_argument('--freeze_encoder', action='store_true')
    p.add_argument('--imagenet_backbone', action='store_true')
    p.add_argument('--ckpt', type=str, default='')
    p.add_argument('--save', type=str, default='/kaggle/working/p3_stage2b_cmf.pth')
    args = p.parse_args()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tr, va, te = make_loaders(args.data_root, batch_size=args.batch_size, n_segments=args.n_segments)

    model = P3Stage2B(n_segments=args.n_segments,
                      freeze_backbone=args.freeze_backbone,
                      freeze_encoder=args.freeze_encoder).to(device)

    # ---- EVAL-ONLY FAST PATH ----
    if args.epochs == 0:
        if not args.ckpt or not os.path.isfile(args.ckpt):
            log_line("ERROR: --epochs 0 requires a valid --ckpt path.")
            return
        log_line(">>> Eval-only: loading checkpoint and running test…")
        load_shape_compatible(model, args.ckpt)
        teL, teA = evaluate(model, te, device)
        log_line(f"Test: loss {teL:.4f} acc {teA:.4f}")
        return

    # ---- TRAINING PATH ----
    if args.imagenet_backbone:
        try:
            log_line("Loading ImageNet weights into ResNet…")
            model.backbone.load_state_dict(resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).state_dict(), strict=False)
            log_line("✓ Loaded.")
        except Exception as e:
            log_line(f"Could not load ImageNet weights: {e}")

    # light unfreeze last resnet block
    unfreeze_layer4(model)
    log_line(">>> Unfroze ResNet.layer4 parameters.")

    if args.ckpt and os.path.isfile(args.ckpt):
        log_line(f"Warm start from {args.ckpt}")
        load_shape_compatible(model, args.ckpt)

    opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr, weight_decay=1e-4)

    best = -1.0
    prev_va = -1.0
    max_extra = 1   # allow +1 extra epoch if last val ↑ >= 0.5 pt
    extra = 0

    log_line(">>> Stage-2B training start")
    try:
        ep = 1
        while ep <= args.epochs + extra:
            trL,trA = train_one_epoch(model, tr, opt, device)
            vaL,vaA = evaluate(model, va, device)
            log_line(f"Epoch {ep:02d} | train {trL:.4f}/{trA:.4f} | valid {vaL:.4f}/{vaA:.4f}")

            # BEST (atomic)
            if vaA > best:
                best = vaA
                atomic_save({"model": model.state_dict()}, args.save)
                log_line(f"  ↳ Saved BEST to {args.save}")
            # EPOCH (atomic)
            ep_path = f"/kaggle/working/p3_stage2b_ep{ep}.pth"
            atomic_save({"model": model.state_dict()}, ep_path)
            log_line(f"  ↳ Saved EPOCH to {ep_path}")

            # make/refresh zip so it's ready in Output
            zip_ckpts()
            os.system("ls -lh /kaggle/working | egrep 'stage2b_.*\\.pth|stage2b_ckpts\\.zip' || true")

            # ---- auto-extend if validation improved by ≥ 0.5 percentage points ----
            if ep == args.epochs and extra < max_extra and (vaA - prev_va) >= 0.005:
                extra += 1
                log_line("↳ Auto-extending training by 1 epoch (val improved ≥ 0.5 pt).")

            prev_va = vaA
            ep += 1

    except Exception as e:
        log_line("!!! Exception occurred, saving last state before exiting.")
        atomic_save({"model": model.state_dict()}, "/kaggle/working/p3_stage2b_last.pth")
        zip_ckpts()
        traceback.print_exc()

    teL,teA = evaluate(model, te, device)
    log_line(f"Test: loss {teL:.4f} acc {teA:.4f}")

if __name__ == "__main__":
    main()


Writing run_stage2b.py


In [5]:
DATA_ROOT = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"
!ls -1 $DATA_ROOT; ls -1 $DATA_ROOT/train; ls -1 $DATA_ROOT/valid; ls -1 $DATA_ROOT/test


test
train
valid
fake
real
fake
real
fake
real


In [ ]:
!python -u run_stage2b.py \
  --data_root $DATA_ROOT \
  --batch_size 16 --epochs 2 --lr 5e-4 \
  --n_segments 50 \
  --freeze_backbone --freeze_encoder \
  --imagenet_backbone \
  --save /kaggle/working/p3_stage2b_cmf.pth


Loading ImageNet weights into ResNet…
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 212MB/s]
✓ Loaded.
>>> Unfroze ResNet.layer4 parameters.
>>> Stage-2B training start
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
Epoch 01 | train 0.1082/0.9588 | valid 0.0164/0.9951
  ↳ Saved BEST to /kaggle/working/p3_stage2b_cmf.pth
  ↳ Saved EPOCH to /kaggle/working/p3_stage2b_ep1.pth
-rw------- 1 root root 170M O

In [ ]:
!zip -j /kaggle/working/stage2b_ckpts.zip /kaggle/working/p3_stage2b_*.pth 2>/dev/null || true
!ls -lh /kaggle/working | egrep 'stage2b_.*(pth|zip)'


In [ ]:
# 1) list checkpoints we saved during training
!ls -lh /kaggle/working | egrep '\.pth|\.log|\.json' || true

# 2) print last 200 lines from the notebook logs, if any
!tail -n 200 /kaggle/working/*.log 2>/dev/null | tail -n 200 || true


In [ ]:
!python -u run_stage2b.py \
  --data_root $DATA_ROOT \
  --epochs 0 \
  --n_segments 50 \
  --freeze_backbone --freeze_encoder \
  --ckpt /kaggle/working/p3_stage2b_cmf.pth


In [ ]:
!ls -lh /kaggle/working | egrep 'stage2b_.*\.pth|stage2b_ckpts\.zip|train_log\.txt' || true
